## Config the model

In [53]:
from langchain_google_genai import ChatGoogleGenerativeAI
model=ChatGoogleGenerativeAI(model='gemini-2.0-flash')

In [54]:
print(model.invoke('hi').content)

Hi there! How can I help you today?


## Config the embedding model

In [55]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings= HuggingFaceEmbeddings(model="BAAI/bge-small-en")
len(embeddings.embed_query("hi"))

384

## Data embedding and store in VDB

In [56]:
from langchain_community.document_loaders import TextLoader,DirectoryLoader
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [57]:
loader=DirectoryLoader("../data2",glob="./*.txt",loader_cls=TextLoader)
#glob means all the txt files

In [58]:
docs=loader.load()

In [59]:
docs[0].page_content

"🇺🇸 Overview of the U.S. Economy\nThe United States of America possesses the largest economy in the world in terms of nominal GDP, making it the most powerful economic force globally. It operates under a capitalist mixed economy, where the private sector dominates, but the government plays a significant regulatory and fiscal role. With a population of over 335 million people and a high level of technological advancement, the U.S. economy thrives on a foundation of consumer spending, innovation, global trade, and financial services. It has a highly diversified structure with strong sectors in technology, healthcare, finance, real estate, defense, and agriculture.\n\nU.S. GDP – Size, Composition, and Global Share\nAs of 2024, the United States’ nominal GDP is estimated to be around $28 trillion USD, accounting for approximately 25% of the global economy. It ranks #1 in the world by nominal GDP, far ahead of China (which ranks 2nd). The U.S. GDP per capita is also among the highest, hover

In [60]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=50)

In [61]:
chunked_data=text_splitter.split_documents(documents=docs)

In [62]:
doc_string=[doc.page_content for doc in chunked_data]

In [63]:
print(doc_string)

['🇺🇸 Overview of the U.S. Economy', 'The United States of America possesses the largest economy in the world in terms of nominal GDP, making it the most powerful economic force globally. It operates under a capitalist mixed economy,', 'It operates under a capitalist mixed economy, where the private sector dominates, but the government plays a significant regulatory and fiscal role. With a population of over 335 million people and a', 'a population of over 335 million people and a high level of technological advancement, the U.S. economy thrives on a foundation of consumer spending, innovation, global trade, and financial services.', 'innovation, global trade, and financial services. It has a highly diversified structure with strong sectors in technology, healthcare, finance, real estate, defense, and agriculture.', 'U.S. GDP – Size, Composition, and Global Share', 'As of 2024, the United States’ nominal GDP is estimated to be around $28 trillion USD, accounting for approximately 25% of

In [64]:
len(chunked_data)

55

In [65]:
db=Chroma.from_documents(chunked_data,embeddings)

In [66]:
retreiver=db.as_retriever(search_kwargs={"k":3})

In [67]:
retreiver.invoke("industrial growth of usa")

[Document(metadata={'source': '..\\data2\\usa.txt'}, page_content='🇺🇸 Overview of the U.S. Economy'),
 Document(metadata={'source': '..\\data2\\usa.txt'}, page_content='🇺🇸 Overview of the U.S. Economy'),
 Document(metadata={'source': '..\\data2\\usa.txt'}, page_content='Looking forward, the U.S. economy is expected to grow at a moderate pace, powered by innovation in AI, green energy, robotics, biotech, and quantum computing. The Biden administration’s Inflation')]

## Creation of pydantic class

In [68]:
from pydantic import BaseModel,Field
from typing import TypedDict,Annotated,Sequence
from langchain_core.messages import BaseMessage
import operator
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import AIMessage,HumanMessage
from langchain.output_parsers import PydanticOutputParser
from langchain_core.output_parsers.string import StrOutputParser

In [69]:
class TopicSelectionParser(BaseModel):
    topic:str=Field(description="selected topic")
    Reasoning:str=Field(description="reasoning behind topic selection")

In [70]:
parser=PydanticOutputParser(pydantic_object=TopicSelectionParser)

In [71]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"topic": {"description": "selected topic", "title": "Topic", "type": "string"}, "Reasoning": {"description": "reasoning behind topic selection", "title": "Reasoning", "type": "string"}}, "required": ["topic", "Reasoning"]}\n```'

In [72]:
'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"topic": {"description": "selected topic", "title": "Topic", "type": "string"}, "Reasoning": {"description": "reasoning behind topic selection", "title": "Reasoning", "type": "string"}}, "required": ["topic", "Reasoning"]}\n```'

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"topic": {"description": "selected topic", "title": "Topic", "type": "string"}, "Reasoning": {"description": "reasoning behind topic selection", "title": "Reasoning", "type": "string"}}, "required": ["topic", "Reasoning"]}\n```'

to add series of messages as a list we use the above class and opertor.add adds more messages

In [73]:
Agentstate={}

In [74]:
Agentstate["messages"]=[]

In [75]:
Agentstate

{'messages': []}

In [76]:
Agentstate["messages"].append("hi how aree you")

In [77]:
Agentstate

{'messages': ['hi how aree you']}

In [78]:
Agentstate["messages"].append("what are you doing")


In [79]:
Agentstate

{'messages': ['hi how aree you', 'what are you doing']}

In [80]:
Agentstate["messages"].append("i hope you are fine")


In [81]:
Agentstate

{'messages': ['hi how aree you', 'what are you doing', 'i hope you are fine']}

In [82]:
Agentstate["messages"][-1]

'i hope you are fine'

In [83]:
Agentstate["messages"][0]

'hi how aree you'

## Agentstate class used in the state graph

In [84]:
class AgentState(TypedDict):
    messages:Annotated[Sequence[BaseMessage],operator.add]

In [85]:
state={"messages":["hi"]}

state will be like the above dictionary

format_instructions (parser instruction)

In [86]:
#Supervisor

def function_1(state:AgentState):

    question=state["messages"][-1]

    print("Question: ",question)

    template='''your task is to classify the given user query into one of the following categories: [USA,not Related] 
    Only respod with the category name and nothing else.

    user query:{question}
    {format_instructions}

    '''
    prompt=PromptTemplate(
        template=template,
        input_variable=["question"],
        partial_variables={"format_instructions":parser.get_format_instructions}
    )
    chain=prompt | model | parser

    response=chain.invoke({"question":question})

    print("Parsed Response: ",response)

    return {"messages":[response.topic]}


In [87]:
state={"messages":["what is the weather today"]}

In [88]:
state={"messages":["what is the gdp of us?"]}

In [89]:
function_1(state)

Question:  what is the gdp of us?
Parsed Response:  topic='USA' Reasoning='The query explicitly asks about the GDP of the United States (US).'


{'messages': ['USA']}

In [90]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

Check the class TopicselectionParser for the above output

In [91]:
#RAG function

def function_2(state:AgentState):
    print("-> RAG Call->")

    question=state["messages"][0]

    prompt=PromptTemplate(
    template='''You are an assistant for question-answering task,use the following
    pieces of retrieved context to answer the question.If you don't know the answer,
    just say you don't know. Use three sentences maximum and keep the the answer
    concise.\nQuestion:{question} \nContext:{context} \nAnswer:''',
    input_type=['context','question']
    )

    rag_chain=(
        {"context": retreiver | format_docs,"question":RunnablePassthrough()}
        |prompt
        |model
        |StrOutputParser()
    )

    result=rag_chain.invoke(question)

    return{"messages":[result]}


In [92]:
#LLM function

def function_3(state:AgentState):
    print("-> LLM Call ->")
    question=state["messages"][0]

    complete_query="answer the question with your knowledge of the real world. following is the user question:"+question
    response=model.invoke(complete_query)

    return{"messages":[response.content]}

In [93]:
#Router

def router(state:AgentState):
    print("-> ROUTER ->")

    last_message=state["messages"][-1]
    print("last_message: ",last_message)
    
    if "usa" in last_message.lower():
        return "RAG Call"
    else:
        return "LLM Call"
    

In [94]:
from langgraph.graph import StateGraph,END

In [95]:
workflow=StateGraph(AgentState)

In [96]:
workflow.add_node("Supervisor",function_1)

In [97]:
workflow.add_node("RAG",function_2)

In [98]:
workflow.add_node("LLM",function_3)

In [99]:
workflow.set_entry_point("Supervisor")

In [100]:
workflow.add_conditional_edges(
    "Supervisor",
    router,{
        "RAG Call":"RAG",
        "LLM Call":"LLM",
    }
)

In [101]:
workflow.add_edge("RAG",END)
workflow.add_edge("LLM",END)

In [102]:
app=workflow.compile()

In [103]:
state={"messages":["what is the gpd of usa"]}

In [104]:
app.invoke(state) #since using the pydantic validation and also it accepts dictionary

Question:  what is the gpd of usa
Parsed Response:  topic='USA' Reasoning='The query explicitly asks about the GDP of the USA.'
-> ROUTER ->
last_message:  USA
-> RAG Call->


{'messages': ['what is the gpd of usa',
  'USA',
  "I am sorry, but the provided context only mentions the overview of the U.S. economy and does not specify the GDP of the USA. Therefore, I don't know the answer."]}

In [108]:
state={"messages":["can you tell me the industrial growth of world's most poor economy"]}

In [109]:
app.invoke(state)

Question:  can you tell me the industrial growth of world's most poor economy
Parsed Response:  topic='not Related' Reasoning="The query asks about the industrial growth of the world's most poor economy, which is not specific to the USA."
-> ROUTER ->
last_message:  not Related
-> LLM Call ->


{'messages': ["can you tell me the industrial growth of world's most poor economy",
  'not Related',
  'It\'s difficult to give a definitive answer to the "industrial growth of the world\'s *most* poor economy" because:\n\n*   **Defining "Most Poor":** There isn\'t a single, universally agreed-upon metric to determine the "most poor" economy. Common indicators include GDP per capita, poverty rates, Human Development Index (HDI), and various measures of inequality. Different metrics can lead to different countries being identified as the poorest.\n*   **Data Availability and Reliability:** The countries with the most impoverished economies often have the least reliable and up-to-date economic data. Official statistics may be incomplete, inaccurate, or subject to political manipulation.\n*   **Informal Sector:** In very poor economies, a significant portion of economic activity occurs in the informal sector, which is difficult to track and measure.\n*   **Fragility and Conflict:** Many o